In [ ]:
"""総合演習2 社会系シミュレーション 第3回 (2026-11-25)

2次元領域を等速直線運動するボールをエージェントとみなしたシミュレーション
"""

import random
from typing import Final

import matplotlib.pyplot as plt
from IPython.display import HTML
from matplotlib import animation
from matplotlib.artist import Artist


# ========== パラメータ ==========
NUM_AGENTS: Final[int] = 10       # エージェントの数
X_MIN: Final[float] = 0           # x座標の最小値
X_MAX: Final[float] = 10          # x座標の最大値
Y_MIN: Final[float] = 0           # y座標の最小値
Y_MAX: Final[float] = 10          # y座標の最大値
VX_ABS: Final[float] = 1          # x方向の速度の絶対値
VY_ABS: Final[float] = 1          # y方向の速度の絶対値

DT: Final[float] = 0.01           # 時間刻み幅
NUM_STEPS: Final[int] = 2000      # 時間ステップ数
PLOT_INTERVAL: Final[int] = 100   # 可視化するステップ間隔
# ==============================


LX: float = X_MAX - X_MIN
LY: float = Y_MAX - Y_MIN
MARGIN: float = 0.1

class MovableAgent:
    """可動エージェントを表すクラス
    
    Attributes
    ----------
    x : float
        エージェントのx座標
    y : float
        エージェントのy座標
    vx : float
        エージェントのx方向の速度
    vy : float
        エージェントのy方向の速度
    """

    def __init__(self, init_x: float, init_y: float, init_vx: float, init_vy: float) -> None:
        """MovableAgentクラスのインスタンスのイニシャライザ

        Parameters
        ----------
        init_x : float
            エージェントの初期x座標
        init_y : float
            エージェントの初期y座標
        init_vx : float
            エージェントのx方向の初期速度
        init_vy : float
            エージェントのy方向の初期速度
        """

        self.x: float = init_x
        self.y: float = init_y
        self.vx: float = init_vx
        self.vy: float = init_vy

    def move(self) -> None:
        """エージェントを1ステップ移動させる
        
        Notes
        -----
        Euler法を使用
        """

        self.x += self.vx * DT
        self.y += self.vy * DT


class MovableAgentSystem:
    """可動エージェント全体を表すクラス
    
    Attributes
    ----------
    num_agents : int
        エージェントの数
    agents : list[MovableAgent]
        可動エージェントのリスト
    """

    def __init__(self, num_agents: int) -> None:
        """MovableAgentSystemクラスのインスタンスのイニシャライザ

        Parameters
        ----------
        num_agents : int
            エージェントの数

        Notes
        -----
        エージェントの初期位置と初期速度はランダムに設定
        """

        self.num_agents: int = num_agents
        self.agents: list[MovableAgent] = []

        for _ in range(num_agents):
            init_x = X_MIN + (X_MAX - X_MIN) * random.random()
            init_y = Y_MIN + (Y_MAX - Y_MIN) * random.random()
            init_vx = VX_ABS * 2*(random.random() - 0.5)
            init_vy = VY_ABS * 2*(random.random() - 0.5)

            self.agents.append(MovableAgent(init_x, init_y, init_vx, init_vy))

    def visualize(self, t: float) -> list[Artist]:
        """各タイムステップでのエージェントの位置を可視化する

        Parameters
        ----------
        t : float
            現在の時刻

        Returns
        -------
        frame : list[Artist]
            可視化結果のArtistオブジェクトのリスト
        """

        frame: list[Artist] = [
            ax.text(X_MIN, Y_MAX, f"t = {t:.1f}", 
                    fontsize=14, backgroundcolor="lightgray")]
        for agent in self.agents:
            frame += ax.plot(agent.x, agent.y, color="blue", marker="o")

        return frame


if __name__ == "__main__":

    system = MovableAgentSystem(NUM_AGENTS)
    t: float = 0

    fig, ax = plt.subplots(figsize=(5, 5), dpi=150)
    ax.set_xlim(X_MIN-MARGIN*LX, X_MAX+MARGIN*LX)
    ax.set_ylim(Y_MIN-MARGIN*LY, Y_MAX+MARGIN*LY)
    ax.grid()

    artists: list[list[Artist]] = []
    artists.append(system.visualize(t))

    for it in range(NUM_STEPS):
        for agent in system.agents:
            agent.move()

        if it % PLOT_INTERVAL == 0:
            artists.append(system.visualize(t))

        t += DT

    anim = animation.ArtistAnimation(fig, artists)
    display(HTML(anim.to_jshtml()))
    plt.close()
    anim.save("./social_3rd_ball2d.gif", writer="pillow")